# 09 — CWE-98 Generalization Study (File Inclusion) on GPT-5.5

**Purpose.** Test whether the two central findings — *Instruction Overload* and *Answer Leakage / hint-leakage* — generalize to a third, mechanistically distinct vulnerability class, **CWE-98 (File Inclusion)**, using the frontier model **GPT-5.5**.

**Why GPT-5.5 (not GPT-4o-mini).** On CWE-95 and CWE-98, GPT-4o-mini exhibits an extreme over-flagging *floor effect*: it labels every sample (including safe ones) as Vulnerable, collapsing Specificity to 0 and erasing all between-prompt differences. This floor effect is itself consistent with the over-flagging tendency reported for GPT-4o-mini in the main study. To obtain a discriminating test of generalization, we therefore use the frontier model GPT-5.5, which a 20-sample probe confirmed achieves 100% correct discrimination on CWE-98 (10/10 vulnerable, 10/10 safe). This places the generalization test within the paper's existing frontier-validation narrative.

**Note on temperature.** The GPT-5.5 API does not support custom temperature (only the default applies); GPT-4o-mini runs in the main study used temperature=0.1. This difference is an API constraint, stated for transparency.

**Conditions (4).** `Variant_A_Baseline`, `Variant_C_Patterns_CLEAN`, `Variant_E_Full_CLEAN`, `Variant_E_Full_HINTED_CWE98`.

**Inputs.** `<BASE_DIR>/final_dataset_cwe98/` (96 vulnerable + 48 safe = 144). **Output.** `<BASE_DIR>/results/cwe98_generalization_gpt55.csv`.

**Cost & runtime.** 144 x 4 = 576 calls on GPT-5.5. Resume-on-interrupt supported.


## 1. Setup

In [1]:
import os, json, time
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Not on Colab; assuming local paths.')

# ---- USER-EDITABLE ----
BASE_DIR = Path('/content/drive/MyDrive/LLM_Security_Paper')
MODEL_ID          = 'gpt-5.5-2026-04-23'
MAX_RETRIES       = 3
RETRY_BACKOFF_SEC = 2
# -----------------------

DATASET_DIR = BASE_DIR / 'final_dataset_cwe98'
RESULTS_DIR = BASE_DIR / 'results'
OUTPUT_CSV  = RESULTS_DIR / 'cwe98_generalization_gpt55.csv'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

api_key = os.environ.get('OPENAI_API_KEY')
if not api_key:
    from getpass import getpass
    api_key = getpass('Enter your OpenAI API key (will not be stored): ')
client = OpenAI(api_key=api_key)

print(f'DATASET_DIR : {DATASET_DIR}  (exists: {DATASET_DIR.exists()})')
print(f'OUTPUT_CSV  : {OUTPUT_CSV}')
print(f'MODEL_ID    : {MODEL_ID}')
if not DATASET_DIR.exists():
    print('\n[WARNING] DATASET_DIR not found. Run the CWE-98 curation step first.')


Mounted at /content/drive
Enter your OpenAI API key (will not be stored): ··········
DATASET_DIR : /content/drive/MyDrive/LLM_Security_Paper/final_dataset_cwe98  (exists: True)
OUTPUT_CSV  : /content/drive/MyDrive/LLM_Security_Paper/results/cwe98_generalization_gpt55.csv
MODEL_ID    : gpt-5.5-2026-04-23


## 2. Prompts (inline, self-contained)

Three prompts are verbatim copies of the committed files; the CWE-98 HINTED prompt mirrors the original HINTED structure but names only CWE-98. The system message is category-neutral (generic "CWE-XX") so the CLEAN conditions stay category-agnostic.

In [2]:
PROMPTS = {
    'Variant_A_Baseline':
        'Analyze the following PHP code. Does it contain any security '
        'vulnerabilities? Return JSON format.',
    'Variant_C_Patterns_CLEAN':
        'Analyze this PHP code. Apply taint analysis to identify data flow from '
        'untrusted user inputs (sources such as $_GET, $_POST, $_REQUEST, $_COOKIE) '
        'to security-sensitive operations (sinks). Determine whether any unsanitized '
        'tainted data reaches a sink. Return JSON.',
    'Variant_E_Full_CLEAN':
        'You are a senior cybersecurity auditor. Think step-by-step: '
        '1. Trace untrusted inputs (sources). 2. Identify security-sensitive '
        'operations (sinks). 3. Evaluate whether sanitization between source and '
        'sink is adequate. Return your final verdict in JSON format.',
    'Variant_E_Full_HINTED_CWE98':
        'You are a senior cybersecurity auditor. Think step-by-step: '
        '1. Trace inputs. 2. Look for CWE-98 patterns. Evaluate sanitization. '
        'Return your final verdict in JSON format.',
}
SYSTEM_MESSAGE = (
    'You must respond strictly in JSON format with exactly three keys:\n'
    '1. "prediction": String, MUST be either "Vulnerable" or "Safe".\n'
    '2. "cwe_type": String, the identified CWE (e.g., "CWE-XX") or "None" if safe.\n'
    '3. "confidence": Integer from 0 to 100.'
)
for name, text in PROMPTS.items():
    print(f'{name:30s} ({len(text):3d}): {text[:65]}...')


Variant_A_Baseline             ( 97): Analyze the following PHP code. Does it contain any security vuln...
Variant_C_Patterns_CLEAN       (263): Analyze this PHP code. Apply taint analysis to identify data flow...
Variant_E_Full_CLEAN           (259): You are a senior cybersecurity auditor. Think step-by-step: 1. Tr...
Variant_E_Full_HINTED_CWE98    (170): You are a senior cybersecurity auditor. Think step-by-step: 1. Tr...


## 3. Run the four conditions on GPT-5.5

Labels inferred from parent directory: `CWE_98_FileIncl` -> Vulnerable (CWE-98); `Safe_Code` -> Safe. Note `temperature` is **not** sent (GPT-5.5 only supports its default). Incremental save; resume-on-interrupt.

In [3]:
def get_true_label(filepath: Path):
    folder = filepath.parent.name
    if 'Safe' in folder:
        return 'Safe', None
    if '98' in folder:
        return 'Vulnerable', 'CWE-98'
    raise ValueError(f'Cannot infer label from folder name: {folder}')

def predict(prompt_text, code_content):
    for attempt in range(MAX_RETRIES):
        try:
            # GPT-5.5 does not accept a custom temperature; omit the parameter.
            response = client.chat.completions.create(
                model=MODEL_ID,
                response_format={'type': 'json_object'},
                messages=[
                    {'role': 'system', 'content': SYSTEM_MESSAGE},
                    {'role': 'user',   'content': f'{prompt_text}\n\nTarget Code:\n{code_content}'},
                ],
            )
            result = json.loads(response.choices[0].message.content)
            return result.get('prediction', 'Error')
        except Exception:
            if attempt + 1 < MAX_RETRIES:
                time.sleep(RETRY_BACKOFF_SEC)
            else:
                return 'Error'

all_files = sorted(DATASET_DIR.rglob('*.php'))
if not all_files:
    raise RuntimeError(f'No .php files under {DATASET_DIR}. Run CWE-98 curation first.')
print(f'Total samples: {len(all_files)}  (expected 144)')

if OUTPUT_CSV.exists():
    results_df = pd.read_csv(OUTPUT_CSV)
    processed = set(results_df['File_Name'].tolist())
    print(f'Resuming: {len(processed)} already processed.')
else:
    results_df = pd.DataFrame(columns=['File_Name','True_Label','True_CWE']+list(PROMPTS.keys()))
    processed = set()

to_process = [p for p in all_files if p.name not in processed]
print(f'Remaining: {len(to_process)}')

for filepath in tqdm(to_process, desc=f'CWE-98 on {MODEL_ID}'):
    tl, tc = get_true_label(filepath)
    content = filepath.read_text(encoding='utf-8', errors='ignore')
    row = {'File_Name': filepath.name, 'True_Label': tl, 'True_CWE': tc}
    for vn, pt in PROMPTS.items():
        row[vn] = predict(pt, content)
    results_df = pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)
    results_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'\nComplete. Written to {OUTPUT_CSV}')


Total samples: 144  (expected 144)
Remaining: 144


CWE-98 on gpt-5.5-2026-04-23:   0%|          | 0/144 [00:00<?, ?it/s]


Complete. Written to /content/drive/MyDrive/LLM_Security_Paper/results/cwe98_generalization_gpt55.csv


## 4. Inspect results

In [4]:
df = pd.read_csv(OUTPUT_CSV)
print(f'Total rows : {len(df)}   (expected 144)')
print()
print('True label distribution:')
print(df['True_Label'].value_counts().to_string())
print()
cols = list(PROMPTS.keys())
print('Per-condition prediction distribution:')
for v in cols:
    print(f'  {v:30s} {df[v].value_counts().to_dict()}')
print()
n_disc = (df[cols].nunique(axis=1) > 1).sum()
print(f'Rows where the four conditions disagree: {n_disc}  (need > 0 to measure effects)')


Total rows : 144   (expected 144)

True label distribution:
True_Label
Vulnerable    96
Safe          48

Per-condition prediction distribution:
  Variant_A_Baseline             {'Vulnerable': 98, 'Safe': 46}
  Variant_C_Patterns_CLEAN       {'Vulnerable': 98, 'Safe': 46}
  Variant_E_Full_CLEAN           {'Vulnerable': 96, 'Safe': 48}
  Variant_E_Full_HINTED_CWE98    {'Vulnerable': 92, 'Safe': 52}

Rows where the four conditions disagree: 12  (need > 0 to measure effects)
